# Using the `tommos-adapter` in Ubermag

This notebook shows how to use the `tommos-adapter` for Ubermag.

In [1]:
import shlex
import subprocess

import micromagneticmodel as mm
import numpy as np
import pyvista as pv

import tommos_adapter as ta

## Generate mesh

We use `tommos` to generate the mesh:

In [2]:
cmd = "tommos mesh --geom box --extent 20,20,20 --h 2 --out-name cube"
subprocess.run(shlex.split(cmd))

[ok] Wrote data: cube.npz (nodes=3284, tets=15109)
[ok] Wrote visualization: cube.vtu (cell_data: mat_id)


CompletedProcess(args=['tommos', 'mesh', '--geom', 'box', '--extent', '20,20,20', '--h', '2', '--out-name', 'cube'], returncode=0)

## Initialize System

We start with a `System` object, with name `hysteresis`.

In [3]:
system = mm.System(name="hysteresis")

## Define magnetization

We assume the mesh is already divided in zones (with names `id`):

In [4]:
state = pv.read("cube.vtu")
id_array = state.get_array("mat_id")
state.cell_data["id"] = id_array  # Rename `mat_id` cell data to `id`
state.cell_data.remove("mat_id")  # remove old data with name `mat_id`
state

UnstructuredGrid (0x7f1b46925300)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   1

We define an initial magnetization, constant $(0, 1/\sqrt(2), 1/\sqrt(2))$

In [5]:
m = np.zeros((state.n_points, 3), dtype=np.float32)
m[:, [1, 2]] = 1 / np.sqrt(2)
m

array([[0.        , 0.70710677, 0.70710677],
       [0.        , 0.70710677, 0.70710677],
       [0.        , 0.70710677, 0.70710677],
       ...,
       [0.        , 0.70710677, 0.70710677],
       [0.        , 0.70710677, 0.70710677],
       [0.        , 0.70710677, 0.70710677]],
      shape=(3284, 3), dtype=float32)

We add the magnetization vector to the state object as vector field

In [6]:
state.point_data["m"] = m
state

UnstructuredGrid (0x7f1b46925300)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   2

Add Spontaneous Polarization property as cell constant

In [7]:
Js = 1.76  # spontaneous polarization in Tesla
Js_array = Js * np.ones_like(state.n_cells, dtype=np.float32)
state.cell_data["Js"] = Js_array
state

UnstructuredGrid (0x7f1b46925300)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   3

We plot the one arrow per cell:

In [8]:
state.point_data_to_cell_data().glyph(orient="m", scale="Js", tolerance=0.05).plot()

Widget(value='<iframe src="http://localhost:44123/index.html?ui=P_0x7f1b3db8d110_0&reconnect=auto" class="pyvi…

 JS Error => no webgl context
 JS Error => TypeError: `target` argument of Proxy must be an object, got null
 JS Error => no webgl context
 JS Error => TypeError: `target` argument of Proxy must be an object, got null


In [9]:
system.m = state

## Define energy terms

Energy terms include intrinsic properties

In [10]:
A = 1e-12  # exchange stiffness constant in J/m
K = 4e5  # first uniaxial anisotropy constant in J/m^3
u = (0, 0, 1)  # anisotropy unit axis
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K, u=u)
system.energy

Exchange(A=1e-12) + UniaxialAnisotropy(K=400000.0, u=(0, 0, 1))

## Define driver and hysteresis parameters

Next, we define the driver. The initialization does not need any argument.

In [11]:
hd = ta.HysteresisDriver()
hd

HysteresisDriver()

Let us simulate hysteresis between $-2$ Tesla and $1$ Tesla applied along the $x$-direction in steps of $0.2$ Tesla:

In [12]:
Hmin = (-2 / mm.consts.mu0, 0, 0)
Hmax = (+1 / mm.consts.mu0, 0, 0)
n = 16

Now we execute the hysteresis loop.

In [13]:
hd.drive(system, Hmin=Hmin, Hmax=Hmax, n=n)

Running Tommos Calculator (TommosRunner)[2026-10-08T17:08:26]... (8.1 s)


In [14]:
system.m

UnstructuredGrid (0x7f1afb90d000)
  N Cells:    15109
  N Points:   3284
  X Bounds:   -1.000e+01, 1.000e+01
  Y Bounds:   -1.000e+01, 1.000e+01
  Z Bounds:   -1.000e+01, 1.000e+01
  N Arrays:   4

In [15]:
system.m.point_data_to_cell_data().glyph(orient="m", scale="Js", tolerance=0.05).plot()

Widget(value='<iframe src="http://localhost:44123/index.html?ui=P_0x7f1afbfd7090_1&reconnect=auto" class="pyvi…